Если окружение еще не настроено:\n\n```python\n# %pip install pandas pyarrow numpy matplotlib seaborn scikit-learn tensorflow\n```

# Team Track: RNN/LSTM baseline-style notebook\n\nНоутбук повторяет структуру `baseline_template.ipynb`, но использует последовательностный подход (LSTM) для многошагового прогноза.

In [ ]:
from dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\nfrom sklearn.preprocessing import StandardScaler\n\nimport tensorflow as tf\nfrom tensorflow.keras import Sequential\nfrom tensorflow.keras.callbacks import EarlyStopping\nfrom tensorflow.keras.layers import Dense, Dropout, LSTM\n

In [ ]:
sns.set_theme(style='whitegrid', context='notebook')\nplt.rcParams['figure.figsize'] = (12, 5)\n\n@dataclass\nclass CFG:\n    DATA_PATH: Path = Path('/Users/lubovklepcova/Desktop/Siriusly/ML/wb/Dd2WPGKz')\n    TRAIN_PATH: str = 'train_team_track.parquet'\n    TEST_PATH: str = 'test_team_track.parquet'\n    TARGET_COL: str = 'target_2h'\n    FORECAST_POINTS: int = 10\n    WINDOW: int = 48\n    MIN_HISTORY: int = 120\n    VALID_RATIO: float = 0.2\n    RANDOM_STATE: int = 42\n    EPOCHS: int = 30\n    BATCH_SIZE: int = 256\n    LR: float = 1e-3\n    LSTM_UNITS: int = 128\n    DROPOUT: float = 0.2\n\ncfg = CFG()\nnp.random.seed(cfg.RANDOM_STATE)\ntf.random.set_seed(cfg.RANDOM_STATE)\n

## Загрузка данных

In [ ]:
train_df = pd.read_parquet(cfg.DATA_PATH / cfg.TRAIN_PATH)\ntest_df = pd.read_parquet(cfg.DATA_PATH / cfg.TEST_PATH)\n\ntrain_df['timestamp'] = pd.to_datetime(train_df['timestamp'])\ntest_df['timestamp'] = pd.to_datetime(test_df['timestamp'])\n\ntrain_df = train_df.sort_values(['route_id', 'timestamp']).reset_index(drop=True)\ntest_df = test_df.sort_values(['route_id', 'timestamp']).reset_index(drop=True)\n\nprint('train shape:', train_df.shape)\nprint('test shape:', test_df.shape)\n

In [ ]:
display(train_df.head())\ndisplay(test_df.head())\n\nprint('Train date range:', train_df['timestamp'].min(), '->', train_df['timestamp'].max())\nprint('Test date range:', test_df['timestamp'].min(), '->', test_df['timestamp'].max())\nprint('Train routes:', train_df['route_id'].nunique())\nprint('Test routes:', test_df['route_id'].nunique())

## EDA train-данных

In [ ]:
overview = pd.DataFrame({\n    'dtype': train_df.dtypes.astype(str),\n    'missing_cnt': train_df.isna().sum(),\n    'missing_pct': (train_df.isna().mean() * 100).round(4),\n    'n_unique': train_df.nunique(dropna=False),\n}).sort_index()\n\noverview

In [ ]:
status_cols = [f'status_{i}' for i in range(1, 9)]\nsample_route = train_df['route_id'].iloc[0]\nroute_slice = train_df.loc[train_df['route_id'] == sample_route, ['timestamp', cfg.TARGET_COL] + status_cols].tail(300)\n\nfig, ax = plt.subplots()\nax.plot(route_slice['timestamp'], route_slice[cfg.TARGET_COL], label=cfg.TARGET_COL)\nax.set_title(f'Route {sample_route}: target dynamics (tail)')\nax.legend()\nplt.show()

## Подготовка supervised dataset для RNN\n\nФормируем окна длины `WINDOW`:\n- `X`: `status_1..status_8 + target_2h` за последние `WINDOW` точек\n- `y`: сразу 10 будущих точек `target_2h` (многошаговый direct forecast)

In [ ]:
feature_cols = status_cols + [cfg.TARGET_COL]\n\ndef build_supervised(df: pd.DataFrame):\n    x_blocks, y_blocks, ts_blocks = [], [], []\n\n    for _, grp in df.groupby('route_id', sort=False):\n        grp = grp.sort_values('timestamp').reset_index(drop=True)\n        if len(grp) < cfg.MIN_HISTORY:\n            continue\n\n        vals = grp[feature_cols].to_numpy(dtype=np.float32)\n        tgt = grp[cfg.TARGET_COL].to_numpy(dtype=np.float32)\n        ts = grp['timestamp'].to_numpy()\n\n        max_start = len(grp) - cfg.WINDOW - cfg.FORECAST_POINTS + 1\n        if max_start <= 0:\n            continue\n\n        for start in range(max_start):\n            end = start + cfg.WINDOW\n            horizon_end = end + cfg.FORECAST_POINTS\n\n            x_blocks.append(vals[start:end])\n            y_blocks.append(tgt[end:horizon_end])\n            ts_blocks.append(ts[end - 1])\n\n    X = np.stack(x_blocks, axis=0)\n    y = np.stack(y_blocks, axis=0)\n    src_ts = pd.to_datetime(np.array(ts_blocks))\n    return X, y, src_ts\n\nX, y, src_ts = build_supervised(train_df)\nprint('X shape:', X.shape)\nprint('y shape:', y.shape)

In [ ]:
split_point = pd.Series(src_ts).quantile(1 - cfg.VALID_RATIO)\nfit_mask = src_ts <= split_point\nvalid_mask = src_ts > split_point\n\nX_fit, y_fit = X[fit_mask], y[fit_mask]\nX_valid, y_valid = X[valid_mask], y[valid_mask]\n\nprint('Fit rows:', X_fit.shape, y_fit.shape)\nprint('Valid rows:', X_valid.shape, y_valid.shape)

## Подготовка test-окон\n\nДля каждого `route_id` берем последние `WINDOW` точек train и прогнозируем следующие 10 точек.

In [ ]:
def build_test_windows(df: pd.DataFrame):\n    route_ids, x_blocks = [], []\n    for route_id, grp in df.groupby('route_id', sort=False):\n        grp = grp.sort_values('timestamp')\n        if len(grp) < cfg.WINDOW:\n            continue\n        x_blocks.append(grp[feature_cols].to_numpy(dtype=np.float32)[-cfg.WINDOW:])\n        route_ids.append(route_id)\n    return np.stack(x_blocks, axis=0), np.array(route_ids, dtype=int)\n\nX_test_raw, test_route_ids = build_test_windows(train_df)\nprint('X_test shape:', X_test_raw.shape)\nprint('routes in test windows:', len(test_route_ids))

In [ ]:
def scale_3d(X_fit, X_valid, X_test):\n    n_fit, seq_len, n_feat = X_fit.shape\n    scaler = StandardScaler()\n    scaler.fit(X_fit.reshape(-1, n_feat))\n\n    X_fit_s = scaler.transform(X_fit.reshape(-1, n_feat)).reshape(n_fit, seq_len, n_feat)\n    X_valid_s = scaler.transform(X_valid.reshape(-1, n_feat)).reshape(X_valid.shape[0], seq_len, n_feat)\n    X_test_s = scaler.transform(X_test.reshape(-1, n_feat)).reshape(X_test.shape[0], seq_len, n_feat)\n    return X_fit_s, X_valid_s, X_test_s\n\nX_fit_s, X_valid_s, X_test_s = scale_3d(X_fit, X_valid, X_test_raw)

## RNN/LSTM модель

In [ ]:
def build_model(seq_len, n_feat, horizon):\n    model = Sequential([\n        LSTM(cfg.LSTM_UNITS, input_shape=(seq_len, n_feat), return_sequences=False),\n        Dropout(cfg.DROPOUT),\n        Dense(64, activation='relu'),\n        Dense(horizon, activation='linear'),\n    ])\n    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=cfg.LR), loss='mae')\n    return model\n\nmodel = build_model(X_fit_s.shape[1], X_fit_s.shape[2], cfg.FORECAST_POINTS)\nmodel.summary()

In [ ]:
callbacks = [\n    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)\n]\n\nhistory = model.fit(\n    X_fit_s,\n    y_fit,\n    validation_data=(X_valid_s, y_valid),\n    epochs=cfg.EPOCHS,\n    batch_size=cfg.BATCH_SIZE,\n    verbose=1,\n    callbacks=callbacks,\n)

## Метрики train/valid (WAPE + |Relative Bias|)

In [ ]:
class WapePlusRBias:\n    @staticmethod\n    def calculate(y_true, y_pred):\n        y_true = y_true.astype(float)\n        y_pred = y_pred.astype(float)\n        den = np.maximum(np.abs(y_true).sum(), 1e-12)\n        wape = np.abs(y_pred - y_true).sum() / den\n        rbias = np.abs((y_pred.sum() - y_true.sum()) / den)\n        return float(wape + rbias)\n\nfit_pred = np.maximum(model.predict(X_fit_s, verbose=0), 0.0)\nvalid_pred = np.maximum(model.predict(X_valid_s, verbose=0), 0.0)\n\nfit_metric = WapePlusRBias.calculate(y_fit, fit_pred)\nvalid_metric = WapePlusRBias.calculate(y_valid, valid_pred)\n\nprint('fit WAPE+|RBias|:', round(fit_metric, 6))\nprint('valid WAPE+|RBias|:', round(valid_metric, 6))

In [ ]:
plt.plot(history.history['loss'], label='train_loss')\nplt.plot(history.history['val_loss'], label='valid_loss')\nplt.title('Training history (MAE)')\nplt.legend()\nplt.show()

## Инференс и формирование submission (как в baseline)

In [ ]:
test_pred = np.maximum(model.predict(X_test_s, verbose=0), 0.0)\ninference_ts = train_df['timestamp'].max()\n\nrows = []\nfor i, route_id in enumerate(test_route_ids):\n    for step in range(1, cfg.FORECAST_POINTS + 1):\n        rows.append({\n            'route_id': int(route_id),\n            'timestamp': inference_ts + pd.Timedelta(minutes=30 * step),\n            'y_pred': float(test_pred[i, step - 1]),\n        })\n\nforecast_df = pd.DataFrame(rows)\nforecast_df = forecast_df.sort_values(['route_id', 'timestamp']).reset_index(drop=True)\n\nsubmission_df = test_df.merge(forecast_df, on=['route_id', 'timestamp'], how='left')[['id', 'y_pred']]\n\nif submission_df['y_pred'].isna().any():\n    route_mean = train_df.groupby('route_id', as_index=True)[cfg.TARGET_COL].mean()\n    submission_df = test_df[['id', 'route_id']].join(\n        test_df['route_id'].map(route_mean).rename('y_pred'),\n        how='left',\n    )[['id', 'y_pred']]\n    submission_df['y_pred'] = submission_df['y_pred'].fillna(train_df[cfg.TARGET_COL].mean())\n\nsubmission_df.head()

In [ ]:
submission_path = 'submission_team_rnn.csv'\nsubmission_df.to_csv(submission_path, index=False)\nprint('submission saved to:', submission_path)